In [1]:
import warnings 
warnings.filterwarnings('ignore')
import langchain_community
from langchain_community.document_loaders import PyPDFLoader
loder = PyPDFLoader('C:\\Users\\SAYAN METE\\OneDrive\\Documents\\Retrival Argumented Generation\\Chapter-10.pdf')
pages = loder.load()


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1200,chunk_overlap = 120)
text = text_spliter.split_documents(pages)
chunks = []
for i in text:
    chunks.append(i.page_content)
metadata = []
for i in text:
    metadata.append(i.metadata)


In [3]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embedding_function = SentenceTransformerEmbeddingFunction('all-MiniLM-L6-v2')

client = chromadb.PersistentClient(path="./hybrid-rag")
collection = client.get_or_create_collection(name='Collection',embedding_function=embedding_function)

try :
    if collection.count()==0:
        collection.add(
            documents = chunks,
            ids = [str(i) for i in range(len(chunks))],       
            metadatas=metadata
        )
except Exception as e:
    print(str(e))

from rank_bm25 import BM25Okapi

def token_create(i):
    i = i.lower()
    i = i.split()
    return i 
token = [token_create(i) for i in chunks]
token_corpus = BM25Okapi(token)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4291.47it/s]


In [4]:
def retrival(query:str)->str:
    query_lower_case = query.lower()

    response = collection.query(query_texts=[query_lower_case],n_results=5)
    document = response['documents'][0]
    distance = response['distances'][0]

    thresold = 1.6
    near_chunks = []
    for i , j in zip(distance,document):
        if thresold>i:
            near_chunks.append(j)

    score = token_corpus.get_scores(token_create(query_lower_case))

    def near_index_find_out(score,k=10):
        index = list(enumerate(score))
        index_sorted = sorted(index, key=lambda x :[1] , reverse=True)
        return [inx for inx , sc in index_sorted[:k]]
    get_index = near_index_find_out(score,k=10)

    index_to_chunk = []
    for i in get_index:
        index_to_chunk.append(chunks[i])

    rrf_item = {}

    for rank , doc in enumerate(near_chunks):
        rrf_item[doc] = rrf_item.get(doc,0) + 1/ (rank+60)
    for rank , doc in enumerate(index_to_chunk):
        rrf_item[doc] = rrf_item.get(doc,0)+1 / (rank+60)

    merge = sorted(rrf_item.items() , key=lambda x :x[1],reverse=True)

    top_docs = []
    for doc, _ in merge[:5]:
        top_docs.append(doc)

    if not top_docs:
        return "NOT RELATED CONTENT"
    return "\n\n".join(top_docs)




In [5]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
api = os.getenv('GROQ_API_KEY')
groq_llm_model = ChatGroq(model='openai/gpt-oss-120b',api_key=api)

question = 'what is force ?'
content = retrival(question)


In [6]:
prompt = """
You are a realiable ai assistent so provide user asking qustions based on only the local document 

content = {content}
qustion = {question}

"""
formatted_prompt = prompt.format(content=content, question=question)
print(groq_llm_model.invoke(formatted_prompt).content)


**Answer (based only on the provided document):**

A **force** is a push or pull that acts on an object.  
- When a force is applied, it can **change the shape** of the object (e.g., pressing an inflated balloon makes it deform).  
- It can also **change the direction** of a moving object (e.g., kicking a football to steer it).  
- Additionally, a force can **change the speed** of a moving object (e.g., applying brakes to a bicycle).  

Thus, force is any interaction that tries to alter an object’s state of rest, its motion, its shape, or its speed.
